In [40]:
#import Libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


from sklearn.cluster import KMeans
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

%matplotlib inline

import warnings
warnings.filterwarnings('ignore')


In [41]:
# Load and Explore thd Data

df = pd.read_csv('UsArrests.csv', index_col=0)

df.head()

,Murder,Assault,UrbanPop,Rape
City,,,,
Alabama,13.2,236,58,21.2
Alaska,10.0,263,48,44.5
Arizona,8.1,294,80,31.0
Arkansas,8.8,190,50,19.5
California,9.0,276,91,40.6


In [42]:
df.info()

<class 'pandas.DataFrame'>
Index: 50 entries, Alabama to Wyoming
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Murder    50 non-null     float64
 1   Assault   50 non-null     int64  
 2   UrbanPop  50 non-null     int64  
 3   Rape      50 non-null     float64
dtypes: float64(2), int64(2)
memory usage: 2.0+ KB


In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

Discussion
The dataset contains crime statistics for the 50 US states:
1) Murder,
2) Assault,
3) UrbanPop,
4) Rape,
No Missing values are present

The variables are measured on diffrent scales:
Assult ranges up to 337.0
Murder ranges up to 17.4
UrbanPor is a percentage

Since PCA is sensetive to scale, standardisation is required before analysis.    

In [ ]:
#Preprocessing

#Standardise the data

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

scaled_df = pd.DataFrame(
    X_scaled,
    columns = df.columns,
    index = df.index
)

scaled_df.head()

In [ ]:
# Correlation Analysis

corr_matrix = df.corr()
plt.figure(figsize=(8,6))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Correlation Matrix')
plt.show()


Interpretation Of Correlations

Strong Positive Correlations
1) Murder and Assault (0.80) : States that with high murder rates generally have high assult rates. This suggests that these voilent crime often occur togather
2) Assult and Rape(0.67): States with higher assult rates tend to also have higher rape rates.

Moderate Positive Correlations
1)Murder and Rate (0.56) : Indiacated a moderate relationsip between these twoo crime types.

Weak Correlations
Urban and Murder (0.07)
Very littel relationship exists between urban population precentage and murder rate.

UrbanPop and Assult (0.26)
Only a weak positive relationship

Overall  Conclusion
Violant crime variables (Murder, Assult and Rape) are stronglly correlated and likely measure similar underlying crime pattern. This suggests PCA should effectively reduce  dimensionality

In [ ]:
# Perform PCA

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

explained_variance = pca.explained_variance_ratio_
explained_variance

In [ ]:
# scree Plot
plt.figure(figsize=(8,5))

plt.plot(
    range(1, len(explained_variance) + 1),
    explained_variance,
    marker='o'
)

plt.xlabel("Principle Component")
plt.ylabel("Explained Variance Ratio")
plt.title('Scree Plot')
plt.grid(True)

plt.show()

In [ ]:
# Cumulative Explained Varience
cumulative_varience = np.cumsum(explained_variance)
plt.figure(figsize=(8,5))


plt.plot(
    range(1, len(cumulative_varience) + 1),
    cumulative_varience,
    marker='o'
)

plt.xlabel('Number Of Components')
plt.ylabel('Cumulative Variene Explained')
plt.title('Cumulative Explained Varience')

plt.axhline(y=0.9, color='r', linestyle='--')

plt.grid(True)
plt.show()

In [ ]:
# Biplot of first Two Principal Components

pca_2 = PCA(n_components = 2)

X_pca2 = pca_2. fit_transform(X_scaled)

loading = pca_2.components_.T

plt.figure(figsize=(10,8))

plt.scatter(
    X_pca2[:,0],
    X_pca2[:,1],
    alpha=0.7
)

for i, state in enumerate(df.index):
    plt.annotate(state,
                 (X_pca2[i,0], X_pca2[i,1]),
                  fontsize=8)

for i, feature in enumerate(df.columns):
    plt.arrow(
        0,0,
        loading[i,0]*3,
        loading[i,1]*3,
        color='red',
        head_width = 0.05
    )

    plt.text(
         loading[i,0]*3.2,
         loading[i,1]*3.2,
         feature,
         color='red'
    )

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA Biplot')

plt.grid(True)
plt.show()

Interpretation of the PCA Bilpot

The biplot displays both the states(Observations) and the orl variables (Murder, Auult, UrbanPop, and Rape) projected onto first two principlal components (PC1 and PC2)

1) Principle Component 1 (PC1)
PC1 appears to be strongly associated with murder, Assults and Rape as the vectors for these variables point predominantly to the right, 
States located on the right-hand side of the plot therefore tend to have higher levels of violent crime.

Example include:
Califormnia
Florida
Nevada
New York

States on the left-hand side such as,
Vermont
North Dakota
Maine
New Hampshire

generally exibit lower violent criime rates.

2) Principle Component 2 (PC2)
PC2 is influenced most strongly by UrbanPop as indicated by the long arrow pointing upward.  States positioned higher on the plot tend to have a leger proportion of their population living in unban areas.

Examples include:
California
New Jersey
Massachusetts
Hawaii

States lower on the plot such as:
Mississippi
North Carolina
South Corolina
have relatively lower urban population percentages

3) Relationship between Variables
The angle between variables vectors indicates their correlation:
1) Murder and Assault point in a very similar direction, indicating a strong positive correlation
2) Assult and Rape also point in similar directions suggesting a positive relationship
3) UrbanPop points in a diffrent direction from the crime variables indicating that urbanisation is less strongly related to violent crime than the crime variables are to one another


This confirms the findings from the correlation matrix, where Murder Assult and Rape showed strong positive correlation matrix, whhere Murder, Assault and Rape showed stong positive correlations

4) State Groupings
Several natural groupings of states can be observed.
High Crime states: cluster on the right side of the plot (e.g California, Florida, Nevada)
Low-crime states: cluster on the left side (e.g. Vermont, north dakota, Maine)
States near the centre have crime rates and urbanisation levels closer to the national average.

Overall Conclusion:
The biplot shows that the first two principal components successfully summarise the structure of the USArrests dataset. PC1 primirily represents overall violent crime levels while PC2 captures diffrences in urban population levels. The strong alignment of the Murder, Assult and Rape vectors demostrates that thses variables contribute similarly to the variation in the data making PCA an effective dimensionality reduction technique for this dataset


In [ ]:
# Slecting Number of Principlal Components

Variance_df = pd.DataFrame({
    'PC': range(1, len(explained_variance)+1),
    'Variance Explained': explained_variance,
    'Cumulative Variance': cumulative_varience
})

Variance_df

Justification

Example sidcussion:

Component  Variance Explained
PC1         0.62
PC2         0.24
PC3         0.89 
PC3         0.04

The first two principle components explain approximately 87% of the total variance.

Since they retain most information while reducing dimensionality from four variables to two, using 2 principal components is appropriate for clustering and visualisation     

In [ ]:
# Clustering Technique 1: K-Means

kmeans = KMeans(
    n_clusters = 3,
    random_state =42,
    n_init = 10 
)

kmeans_labels = kmeans.fit_predict(X_pca2)

plt.figure(figsize=(8,6))

plt.scatter(
    X_pca2[:,0],
    X_pca2[:,1],
    c= kmeans_labels,
    cmap='viridis'
)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('K-Means Clustering')

plt.show()

# K-Means Interpritation
Typical clusters:
Cluster 1
States with:
High murder
High Assult
High Rape

Example:
California
Florida
Nevada


Cluster 2
Moderate Crime states

Cluster 3
Low Crime states

examples
North Dakota
Vermont
Maine

In [ ]:
# Clustering Technique 2 Hierarchical Clustering

linked = linkage(X_pca2, method='ward')

plt.figure(figsize=(12,6))

dendrogram(
    linked,
    labels= df.index.tolist(),
    leaf_rotation=90
)

plt.title('Hierarchical Clustering Dnedrogram')
plt.xlabel('States')
plt.ylabel('Distance')

plt.show()

In [ ]:
#Generate Clusters

hc = AgglomerativeClustering(
    n_clusters  = 3,
    linkage= 'ward'
)

hc_labels = hc.fit_predict(X_pca2)

plt.figure(figsize=(8,6))

plt.scatter(
    X_pca2[:,0],
    X_pca2[:,1],
    c= hc_labels,
    cmap= 'rainbow'
)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Hiearachical Clustering')

plt.show()

In [ ]:
# Cluster Analysis

cluster_df = df.copy()

cluster_df['KMeans'] = kmeans_labels
cluster_df['Hierarchical'] = hc_labels

cluster_df.groupby('KMeans').mean()

In [ ]:

print("K-means Cluster Profiles")
display(cluster_df.groupby('KMeans').mean())

print("\n Hirerarchical Clustering Cluster Profiles")
display(cluster_df.groupby('Hierarchical').mean())

# Discussion of Clusters

High Crime Cluster

Characteristics:
High Murder rates
High Assult rates
High Rape rates

represents states with the most severe violent crime problems


Medium-Crime Cluster
Charactetristics
Moderate values accross all crime indicators
Represent Trasitional states

Low Crime Cluster
Characteristics:
Low Murder
Low Assult
Low Rape

Represents the safest states in the dataset

# Comparision of Clustering Methods

Similatities
Both K-Means and Hierarchical Clustering separate high-crime and low-crime states clearly
States with similar patterns tend to remain grouped togather

Diffrences
Hierarchical clustering may split some borderline states differently because it uses a nested grouping structure
K-Means forces spherical clusters and assigns every state to the nearest centroid

Both clustering techniques produce similar overall pattrens suggetsting that the PCA representation captures the main structure of the data effectively.
The clusters largely correspond to low-, medium, and high-crime states, demostrating meanngful segmentation of the USArrests dataset.